# MODULE: flake_derivedtypes

## Description
Derived types (data structures) for FLake model  
Converted from: `flake_derivedtypes.f90`

## Original Code Owner
DWD, Dmitrii Mironov

## History
- Version 1.00 (2005/11/17) - Initial release

## Role
Defines FLake's derived types/data structures.

## Defines
- `nband_optic_max` - Maximum number of wavelength bands (10)
- `OpticparMedium` - Dataclass for optical properties of medium (water/ice/snow)

## Dependencies
- **Uses:** `data_parameters`
- **Used by:** `flake_paramoptic_ref`, `flake_core`, `flake_interface`

In [ ]:
import numpy as np
from dataclasses import dataclass

# Import from data_parameters module
# When running standalone, define locally
try:
    from data_parameters import ireals, iintegers
except ImportError:
    ireals = np.float64
    iintegers = np.int32
    print("⚠ Running standalone - data_parameters not imported")

print("FLake Model - MODULE: flake_derivedtypes")
print("=" * 70)

## Maximum Number of Optical Bands

Storage for a ten-band approximation is allocated, although a smaller number of bands is typically used in practice.

In [ ]:
# Maximum number of wave-length bands in the exponential decay law
# for the radiation flux penetration
nband_optic_max = iintegers(10)

## OpticparMedium Dataclass

This dataclass represents optical properties used for radiation penetration calculations.

### Radiation Model

Solar radiation penetrating through a medium (water, ice, or snow) is modeled as:

$$
I(z) = \sum_{i=1}^{n_{bands}} f_i \cdot I_0 \cdot e^{-k_i \cdot z}
$$

Where:
- $I(z)$ = radiation intensity at depth $z$
- $f_i$ = fraction of radiation in band $i$ (frac_optic)
- $k_i$ = extinction coefficient for band $i$ (extincoef_optic) [m⁻¹]
- $n_{bands}$ = number of active bands (nband_optic)

In [ ]:
@dataclass
class OpticparMedium:
    """
    Optical parameters for radiation penetration in water medium.
    
    This class represents the optical characteristics used to calculate
    how solar radiation penetrates and is absorbed in the water column.
    The radiation flux is modeled as a sum of exponential decay functions,
    each representing a wavelength band.
    
    Attributes
    ----------
    nband_optic : np.int32
        Number of wave-length bands actually used (1 to nband_optic_max)
    frac_optic : np.ndarray
        Shape: (10,), dtype: np.float64
        Fractions of total radiation flux for each wavelength band
        Sum of all fractions should equal 1.0
    extincoef_optic : np.ndarray
        Shape: (10,), dtype: np.float64
        Extinction coefficients [m^-1] for each wavelength band
        Larger values indicate stronger absorption/scattering
    
    Example
    -------
    For a two-band approximation (visible + infrared):
    
    >>> optic = OpticparMedium(
    ...     nband_optic=2,
    ...     frac_optic=[0.4, 0.6, 0, 0, 0, 0, 0, 0, 0, 0],
    ...     extincoef_optic=[0.2, 2.0, 0, 0, 0, 0, 0, 0, 0, 0]
    ... )
    # 40% visible (slow absorption), 60% IR (fast absorption)
    """
    nband_optic: iintegers
    frac_optic: np.ndarray  # shape (10,), dtype ireals
    extincoef_optic: np.ndarray  # shape (10,), dtype ireals
    
    def __post_init__(self):
        """Validate and ensure correct array types after initialization."""
        # Ensure arrays have correct dtype and shape
        if not isinstance(self.frac_optic, np.ndarray):
            self.frac_optic = np.array(self.frac_optic, dtype=ireals)
        if not isinstance(self.extincoef_optic, np.ndarray):
            self.extincoef_optic = np.array(self.extincoef_optic, dtype=ireals)
        
        # Ensure correct shape
        assert self.frac_optic.shape == (nband_optic_max,), \
            f"frac_optic must have shape ({nband_optic_max},)"
        assert self.extincoef_optic.shape == (nband_optic_max,), \
            f"extincoef_optic must have shape ({nband_optic_max},)"
        
        # Ensure correct dtype
        if self.frac_optic.dtype != ireals:
            self.frac_optic = self.frac_optic.astype(ireals)
        if self.extincoef_optic.dtype != ireals:
            self.extincoef_optic = self.extincoef_optic.astype(ireals)
    
    def validate(self) -> bool:
        """
        Validate the optical parameters.
        
        Returns
        -------
        bool
            True if valid, raises AssertionError otherwise
        
        Raises
        ------
        AssertionError
            If any validation check fails
        """
        # Check band count is within valid range
        assert 1 <= self.nband_optic <= nband_optic_max, \
            f"nband_optic must be between 1 and {nband_optic_max}"
        
        # Check that fractions sum to 1.0 for active bands
        total_frac = np.sum(self.frac_optic[:self.nband_optic])
        assert np.abs(total_frac - 1.0) < 1e-6, \
            f"Sum of frac_optic for active bands must equal 1.0, got {total_frac}"
        
        # Check that extinction coefficients are positive for active bands
        assert np.all(self.extincoef_optic[:self.nband_optic] > 0), \
            "Extinction coefficients must be positive for active bands"
        
        return True

## Module Verification

In [ ]:
print("Derived Types Module Loaded")
print("-" * 70)
print(f"nband_optic_max: {nband_optic_max}")
print(f"OpticparMedium dataclass defined")
print("=" * 70)

# Test example
print("\nTest Example: Single-band approximation")
print("-" * 70)
test_optic = OpticparMedium(
    nband_optic=iintegers(1),
    frac_optic=np.array([1.0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=ireals),
    extincoef_optic=np.array([0.3, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=ireals)
)
print(f"  nband_optic: {test_optic.nband_optic}")
print(f"  frac_optic[0]: {test_optic.frac_optic[0]}")
print(f"  extincoef_optic[0]: {test_optic.extincoef_optic[0]} m^-1")
print(f"  Validation: {test_optic.validate()}")
print("\n✅ flake_derivedtypes module ready")

## Module Status

✅ **Module complete and ready for use**

This module can now be imported by other FLake modules:
```python
from flake_derivedtypes import nband_optic_max, OpticparMedium
```